In [2]:
import os
from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv()

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
model="claude-sonnet-4-5"

In [ ]:
# helper functions
def add_user_message(messages,text):
    user_message = {"role": "user", "context": text}
    messages.append(user_message)

def add_assistant_message(messages,text):
    assistant_message = {"role": "assistant", "context": text}
    messages.append(assistant_message)


def chat(messages, system = None, temperature=1.0, stop_sequences=[] ):
    params={
        "model" :model,
        "max_tokens" : 1000,
        "messages":messages,
        "stop_sequences" : stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    return message.content[0].text

In [6]:
# Tools and Schemas

# tool 1:- Get the current date time
#step 1:- of alling tool function :- write tool function
from datetime import datetime, timedelta

from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

#step 2:- write JSON Schema
get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [8]:
# step 3:- Call calude with json schema

messages = []

messages.append({
    "role":"user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})


response = client.messages.create(
    model= model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

messages.append({ 
    "role": "assistant",
    "content" : response.content
})

messages


[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_019gThD6eXXExWUv2wBVaknr', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]